# Newton's Method: Root-Finding and Machine Learning Optimization

This notebook explores **Newton's method** (also known as the Newton-Raphson method), one of the most powerful and widely used iterative algorithms in applied mathematics and machine learning.

## Outline

1. **Geometric Intuition & Derivation:** Root finding using tangent line linearizations.
2. **Square Root Algorithm (Babylonian Method):** Computing $\sqrt{N}$ from first principles.
3. **Visualization:** Plotting successive tangent lines converging to the root of $f(x) = x^2 - 9$.
4. **General Newton-Raphson Solver:** Implementation with convergence checks and error tolerance.
5. **Connection to Machine Learning:** From finding roots ($f(x) = 0$) to optimizing objective functions ($L'(\theta) = 0$).


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = [7.0, 4.5]
plt.rcParams["figure.autolayout"] = True

## 1. Geometric Derivation of Newton's Method

Suppose we wish to solve the nonlinear equation:
$$f(x) = 0$$

Given a current estimate $x_n$, we approximate $f(x)$ locally using the tangent line at $(x_n, f(x_n))$:
$$y - f(x_n) = f'(x_n)(x - x_n)$$

To find the next estimate $x_{n+1}$, we locate where this tangent line intersects the $x$-axis ($y = 0$):
$$0 - f(x_n) = f'(x_n)(x_{n+1} - x_n)$$

Solving for $x_{n+1}$:
$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$


## 2. Special Case: Computing Square Roots (The Babylonian Algorithm)

To approximate $\sqrt{N}$, let:
$$f(x) = x^2 - N = 0, \qquad f'(x) = 2x$$

Applying the Newton-Raphson update:
$$x_{n+1} = x_n - \frac{x_n^2 - N}{2x_n} = \frac{2x_n^2 - x_n^2 + N}{2x_n} = \frac{x_n^2 + N}{2x_n} = \frac{1}{2}\left(x_n + \frac{N}{x_n}\right)$$

Both expressions are mathematically identical. Let us verify this in code.


In [ ]:
def sqrt_newton_general(n_val: float, num_loops: int, x0: float | None = None) -> float:
    """General Newton-Raphson: x_{n+1} = x_n - f(x_n) / f'(x_n)."""
    x_n = n_val if x0 is None else x0
    for _i in range(num_loops):
        f_val = x_n**2 - n_val
        df_val = 2.0 * x_n
        x_next = x_n - f_val / df_val
        x_n = x_next
    return x_n


def sqrt_babylonian(n_val: float, num_loops: int, x0: float | None = None) -> float:
    """Simplified Babylonian update: x_{n+1} = (x_n + n_val / x_n) / 2."""
    x_n = n_val if x0 is None else x0
    for _i in range(num_loops):
        x_next = (x_n + n_val / x_n) / 2.0
        x_n = x_next
    return x_n


# Verification on N = 9 and N = 2
print("Computing sqrt(9) with 6 loops:")
print("  General Newton:    ", sqrt_newton_general(9.0, 6))
print("  Babylonian formula:", sqrt_babylonian(9.0, 6))
assert abs(sqrt_newton_general(9.0, 6) - 3.0) < 1e-12
assert abs(sqrt_babylonian(9.0, 6) - 3.0) < 1e-12

print("\nComputing sqrt(2) with 6 loops:")
res = sqrt_babylonian(2.0, 6, x0=2.0)
print(f"  Approx: {res:.15f}")
print(f"  Truth:  {math.sqrt(2.0):.15f}")
print(f"  Error:  {abs(res - math.sqrt(2.0)):.2e}")

### Quadratic Convergence

Notice how rapidly Newton's method converges. Once in the neighborhood of the root, the error satisfies:
$$|e_{n+1}| \le M |e_n|^2$$
This means the number of correct decimal digits roughly **doubles** with each iteration!


In [ ]:
x_val = 2.0  # initial guess for sqrt(2)
true_sqrt2 = math.sqrt(2.0)

print(f"{'Iter':>4} | {'Estimate':>18} | {'Absolute Error':>16} | {'Correct Digits':>14}")
print("-" * 60)

for it in range(1, 7):
    x_val = (x_val + 2.0 / x_val) / 2.0
    err = abs(x_val - true_sqrt2)
    correct_digits = -math.log10(max(err, 1e-16))
    print(f"{it:4d} | {x_val:18.15f} | {err:16.2e} | {correct_digits:14.1f}")

## 3. Visualization: Tangent Lines Converging to Root of $f(x) = x^2 - 9$

We visualize the geometry: at each step, we construct the tangent line at $(x_n, f(x_n))$ and follow it down to its $x$-intercept $x_{n+1}$.


In [ ]:
def f_example(x):
    return x**2 - 9.0


def df_example(x):
    return 2.0 * x


def tangent_line(m, x_pt, y_pt, x):
    return m * (x - x_pt) + y_pt


# Domain for plot
x_grid = np.linspace(-1.0, 6.0, 200)
y_grid = f_example(x_grid)

plt.figure(figsize=(9, 6))
plt.plot(x_grid, y_grid, "k-", lw=2, label=r"$f(x) = x^2 - 9$")
plt.axhline(0, color="gray", linestyle=":", alpha=0.7)
plt.axvline(0, color="gray", linestyle=":", alpha=0.7)

# Trace 3 iterations starting from x0 = 1.0
current_x = 1.0
colors = ["red", "orange", "green"]

for step, col in enumerate(colors):
    current_y = f_example(current_x)
    slope = df_example(current_x)
    next_x = current_x - current_y / slope

    # Plot tangent line segment
    x_tangent = np.linspace(min(current_x, next_x) - 0.5, max(current_x, next_x) + 0.5, 50)
    y_tangent = tangent_line(slope, current_x, current_y, x_tangent)
    plt.plot(
        x_tangent,
        y_tangent,
        linestyle="--",
        color=col,
        alpha=0.8,
        label=f"Step {step + 1}: x_{step}={current_x:.2f} -> x_{step + 1}={next_x:.2f}",
    )

    # Plot point on curve and intercept
    plt.plot([current_x], [current_y], "o", color=col)
    plt.plot([current_x, current_x], [0, current_y], ":", color=col, alpha=0.5)
    plt.plot([next_x], [0], "x", color=col, markersize=8)

    current_x = next_x

plt.plot([3.0], [0.0], "m*", markersize=14, label="Root x = 3.0")
plt.title("Newton-Raphson Iterations for $f(x) = x^2 - 9$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.ylim(-12, 20)
plt.xlim(-1, 6)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. General Robust Newton-Raphson Implementation

In production numerical libraries, Newton's method includes:
1. Convergence tolerance on successive steps: $|x_{n+1} - x_n| < \text{tol}$.
2. Convergence tolerance on function residual: $|f(x_{n+1})| < \text{tol}$.
3. Zero-derivative guard to prevent division by zero.
4. Maximum iteration cutoff to prevent infinite loops when oscillating.


In [ ]:
def newton_raphson(
    f, df, x0: float, tol: float = 1e-10, max_iter: int = 100
) -> tuple[float, list[float]]:
    """Robust 1D Newton-Raphson root finder.

    Returns:
        (root, history_of_estimates)
    """
    history = [x0]
    x_curr = x0

    for _iteration in range(max_iter):
        f_val = f(x_curr)
        df_val = df(x_curr)

        if abs(df_val) < 1e-14:
            raise ZeroDivisionError(f"Derivative too close to zero at x = {x_curr}")

        x_next = x_curr - f_val / df_val
        history.append(x_next)

        if abs(x_next - x_curr) < tol and abs(f(x_next)) < tol:
            return x_next, history

        x_curr = x_next

    return x_curr, history


# Test on cubic polynomial: f(x) = x^3 - 2x - 5 (Real root ≈ 2.0945514815)
def f_poly(x):
    return x**3 - 2.0 * x - 5.0


def df_poly(x):
    return 3.0 * x**2 - 2.0


root, hist = newton_raphson(f_poly, df_poly, x0=3.0)
print(f"Cubic root found: {root:.10f} in {len(hist) - 1} iterations")
print(f"Residual f(root): {f_poly(root):.2e}")
assert abs(f_poly(root)) < 1e-10

## 5. From Root-Finding to Optimization in Machine Learning

How does root-finding relate to training machine learning models?

When training a model with parameters $\theta$ and loss function $\mathcal{L}(\theta)$, our goal is to find the minimum of $\mathcal{L}(\theta)$.
At a local minimum, the first derivative (gradient) must vanish:
$$\mathcal{L}'(\theta) = 0$$

Applying Newton's method to find the root of $g(\theta) = \mathcal{L}'(\theta) = 0$ yields **Newton's Optimization Method**:
$$\theta_{n+1} = \theta_n - \frac{\mathcal{L}'(\theta_n)}{\mathcal{L}''(\theta_n)}$$

### Comparison with Gradient Descent

| Property | Gradient Descent (First-Order) | Newton's Method (Second-Order) |
| :--- | :--- | :--- |
| **Update rule** | $\theta_{n+1} = \theta_n - \eta \mathcal{L}'(\theta_n)$ | $\theta_{n+1} = \theta_n - \frac{\mathcal{L}'(\theta_n)}{\mathcal{L}''(\theta_n)}$ |
| **Information used** | First derivative (slope) | First and second derivatives (curvature) |
| **Step size $\eta$** | Requires manual learning rate tuning | Automatically adapts step size via curvature |
| **Convergence rate** | Linear | Quadratic |
| **Multi-dimensional cost** | $\mathcal{O}(D)$ per step | $\mathcal{O}(D^3)$ to invert $D \times D$ Hessian matrix |


In [ ]:
# Demonstrate 1D optimization of a loss function
# L(theta) = theta^4 - 4*theta^2 + theta
def loss(theta):
    return theta**4 - 4.0 * theta**2 + theta


def dloss(theta):
    return 4.0 * theta**3 - 8.0 * theta + 1.0


def d2loss(theta):
    return 12.0 * theta**2 - 8.0


# Optimize with Newton's method
theta = 2.0
print(f"Initial theta = {theta:.4f} | Loss = {loss(theta):.4f}")
for i in range(5):
    step = dloss(theta) / d2loss(theta)
    theta -= step
    print(
        f"Iter {i + 1}: theta = {theta:10.7f} | Loss = {loss(theta):10.7f} | dL = {dloss(theta):10.2e}"
    )

# Minimum should have dloss == 0
assert abs(dloss(theta)) < 1e-6
print(f"Successfully converged to local minimum at theta = {theta:.6f}!")

## Summary

1. Newton's method replaces a nonlinear curve with its local tangent line, yielding the update $x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$.
2. For quadratic functions and root-finding, convergence is quadratic: error is squared at each step, doubling the number of accurate digits.
3. In machine learning, Newton's method optimizes objective functions by solving for stationary points $\mathcal{L}'(\theta) = 0$, utilizing second-order curvature information ($L''(\theta)$ or the Hessian matrix).
